<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/rec6_x_vidsim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. Package acceleration suite
!pip install -q rdkit mdanalysis pandas

# 2. Binary compilation retrieval
!wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina
!chmod +x gnina

print("==========================================================")
print("✅ NEW NOTEBOOK STORAGE ENVIRONMENT VERIFIED")
print("==========================================================")
print("• Receptor Template Target : receptor6.pdb")
print("• Active Screening Library : vidasim.sdf")
print("• GNINA Neural Driver     : Live")

✅ NEW NOTEBOOK STORAGE ENVIRONMENT VERIFIED
• Receptor Template Target : receptor6.pdb
• Active Screening Library : vidasim.sdf
• GNINA Neural Driver     : Live


In [4]:
import os
import sys
import math
import shutil
import subprocess
import numpy as np
import pandas as pd
from rdkit import Chem
from google.colab import files

# ==========================================
# 1. CORE CONFIGURATION & ENVIRONMENT SETUP
# ==========================================
RECEPTOR = "receptor6.pdb"
INPUT_LIBRARY = "vidasim.sdf"
TARGET_RESIDUES_STR = "resid 1148 1163 1167 1182"

BATCH_SIZE = 50
CONTACT_CUTOFF = 4.0
CHECKPOINT_DIR = "receptor6_checkpoints"

print("==========================================================")
print("🎬 INITIALIZING END-TO-END AUTONOMOUS PIPELINE: RECEPTOR-3")
print("==========================================================")

# Install structural biology libraries dynamically if missing
try:
    import MDAnalysis as mda
except ImportError:
    print("• Installing core topological suites...")
    subprocess.run("pip install -q rdkit mdanalysis pandas", shell=True)
    import MDAnalysis as mda

# Download and permission verification for GNINA binary framework
if not os.path.exists("gnina"):
    print("• Downloading GNINA structural inference engine...")
    subprocess.run("wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina", shell=True)
subprocess.run("chmod +x gnina", shell=True)

# Workspace Asset Protection
if not os.path.exists(RECEPTOR) or not os.path.exists(INPUT_LIBRARY):
    raise FileNotFoundError(f"❌ Error: Ensure {RECEPTOR} and {INPUT_LIBRARY} are present in your workspace sidebar!")

# ==========================================
# 2. STEP 2: DYNAMIC CAVITY GEOMETRY MAPPING
# ==========================================
print("\n➔ Mapping 3D spatial boundaries from target amino acid network...")
u_rec = mda.Universe(RECEPTOR)
pocket_atoms = u_rec.select_atoms(TARGET_RESIDUES_STR)

if len(pocket_atoms) == 0:
    raise ValueError(f"❌ Selection Error: Targeted residues could not be mapped within {RECEPTOR}!")

# Automatically resolve center points and safe bounding padding arrays
CX, CY, CZ = pocket_atoms.center_of_mass()
positions = pocket_atoms.positions
coordinate_span = np.max(positions, axis=0) - np.min(positions, axis=0)

BOX_SIZE = math.ceil(np.max(coordinate_span) + 10.0)
if BOX_SIZE < 16: BOX_SIZE = 16

print(f"   • Tracked Entities : {len(pocket_atoms.residues)} Residues (Phe1148, Asp1163, Gly1167, Glu1182)")
print(f"   • Resolved Center  : X = {CX:.3f} | Y = {CY:.3f} | Z = {CZ:.3f}")
print(f"   • Boundary Envelope: {BOX_SIZE} Å Bounding Cube")

# ==========================================
# 3. STEP 3: HIGH-SPEED PARALLEL DOCKING ENGINE
# ==========================================
supplier = Chem.SDMolSupplier(INPUT_LIBRARY)
molecules = [m for m in supplier if m is not None]
total_mols = len(molecules)
total_batches = math.ceil(total_mols / BATCH_SIZE)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_outputs = []

print(f"\n➔ Deploying GPU accelerated screening routines over {total_mols} compounds...")

for b in range(total_batches):
    batch_sdf = f"{CHECKPOINT_DIR}/batch_{b+1}_input.sdf"
    batch_out = f"{CHECKPOINT_DIR}/batch_{b+1}_docked.sdf"
    checkpoint_outputs.append(batch_out)

    # Write optimized local chunk to avoid memory leaks
    writer = Chem.SDWriter(batch_sdf)
    for m in molecules[b*BATCH_SIZE : (b+1)*BATCH_SIZE]:
        writer.write(m)
    writer.close()

    if os.path.exists(batch_out) and os.path.getsize(batch_out) > 0:
        continue

    print(f"   • Syncing Batch Block {b+1}/{total_batches}...")

    # Execution matrix with rapid screening adjustments (Exhaustiveness 2, Modes 1)
    gnina_cmd = (
        f"./gnina -r {RECEPTOR} -l {batch_sdf} "
        f"--center_x {CX} --center_y {CY} --center_z {CZ} "
        f"--size_x {BOX_SIZE} --size_y {BOX_SIZE} --size_z {BOX_SIZE} "
        f"--num_modes 1 --exhaustiveness 2 --device 0 "
        f"--out {batch_out} --cnn_scoring rescore"
    )
    subprocess.run(gnina_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✅ Trajectory checkpoints stabilized on disk.")

# ==========================================
# 4. STEP 4: MECHANISM CONTACT MATRIX EXTRACTOR
# ==========================================
print("\n➔ Running atomistic Euclidean proximity maps...")
target_coords = pocket_atoms.positions
valid_site_binders = []
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for file_path in docked_files:
    supp = Chem.SDMolSupplier(file_path)
    for mol in supp:
        if mol is None: continue

        conf = mol.GetConformer()
        ligand_coords = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])

        # Matrix calculation between ligand and target atoms
        distances = np.linalg.norm(ligand_coords[:, np.newaxis, :] - target_coords[np.newaxis, :, :], axis=2)
        min_overall_distance = np.min(distances)

        if min_overall_distance <= CONTACT_CUTOFF:
            mol_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(valid_site_binders)}"
            valid_site_binders.append({
                "mol_obj": mol,
                "Identifier": mol_name,
                "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
                "CNN_Score": float(mol.GetProp('CNNscore')),
                "CNN_Affinity": float(mol.GetProp('CNNaffinity')),
                "Proximity_Å": round(min_overall_distance, 3)
            })

OUTPUT_LEADS_DIR = "Receptor6_Top_5_Complexes"
os.makedirs(OUTPUT_LEADS_DIR, exist_ok=True)
os.makedirs("Discovery_Studio_Ready_Plots", exist_ok=True)
os.makedirs("Vina_Energy_Top_Hits", exist_ok=True)

with open(RECEPTOR, 'r') as f:
    receptor_text = f.read()

if len(valid_site_binders) == 0:
    print("⚠️ Warning: No compounds mapped within the 4.0 Å boundary threshold of target residues.")
    top_5_leads = pd.DataFrame()
else:
    df_filtered = pd.DataFrame(valid_site_binders)
    # Sort ascending for lowest thermodynamic free energy values
    df_filtered = df_filtered.sort_values(by="Vina_Energy", ascending=True).reset_index(drop=True)
    top_5_leads = df_filtered.head(5)

    # Export elite structural leads
    for idx, row in top_5_leads.iterrows():
        clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
        with open(f"{OUTPUT_LEADS_DIR}/Absolute_Rank_{idx+1}_{clean_id}_complex.pdb", 'w') as out:
            out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

    # Compile additional comparison perspectives
    cutoff_count = max(1, int(math.ceil(len(df_filtered) * 0.05)))
    df_cnn = df_filtered.sort_values(by="CNN_Score", ascending=False).head(cutoff_count)
    for idx, row in df_cnn.iterrows():
        with open(f"Discovery_Studio_Ready_Plots/CNN_Rank_{idx+1}_complex.pdb", 'w') as out:
            out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

    df_vina = df_filtered.sort_values(by="Vina_Energy", ascending=True).head(cutoff_count)
    for idx, row in df_vina.iterrows():
        with open(f"Vina_Energy_Top_Hits/Vina_Rank_{idx+1}_complex.pdb", 'w') as out:
            out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

print("\n==========================================================")
print(" 🏆 RECEPTOR-6 LEAD DISCOVERY BOARD (TOP 5)")
print("==========================================================")
if not top_5_leads.empty:
    print(top_5_leads[["Identifier", "Vina_Energy", "CNN_Score", "CNN_Affinity", "Proximity_Å"]].to_string(index=False))
else:
    print("No leads matched selection parameters.")
print("-"*60)

# ==========================================
# 5. STEP 5: FILE SYSTEM MASTER ARCHIVAL & METRIC STREAM
# ==========================================
MASTER_ZIP_NAME = "Receptor3_Screening_Workspace_Master"
staging_dir = "master_automation_staging_r6"
os.makedirs(staging_dir, exist_ok=True)

WORKSPACE_ITEMS = [
    OUTPUT_LEADS_DIR,
    "Discovery_Studio_Ready_Plots",
    "Vina_Energy_Top_Hits",
    CHECKPOINT_DIR,
    "gnina",
    RECEPTOR,
    INPUT_LIBRARY
]

print("\n➔ Packaging structural frameworks into unified master volume...")
for item in WORKSPACE_ITEMS:
    if os.path.exists(item):
        dest = os.path.join(staging_dir, item)
        if os.path.isdir(item):
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)

shutil.make_archive(MASTER_ZIP_NAME, 'zip', staging_dir)
shutil.rmtree(staging_dir)

print("\n⚡ Deploying automated browser download stream...")
print("==========================================================")
files.download(f"{MASTER_ZIP_NAME}.zip")
print("🎉 Master routine complete. Extract package locally for structural visualization.")

🎬 INITIALIZING END-TO-END AUTONOMOUS PIPELINE: RECEPTOR-3



➔ Mapping 3D spatial boundaries from target amino acid network...
   • Tracked Entities : 4 Residues (Phe1148, Asp1163, Gly1167, Glu1182)
   • Resolved Center  : X = 197.946 | Y = 186.683 | Z = 170.782
   • Boundary Envelope: 93 Å Bounding Cube


[20:34:56] WARNING: not removing hydrogen atom without neighbors



➔ Deploying GPU accelerated screening routines over 502 compounds...
   • Syncing Batch Block 1/11...


KeyboardInterrupt: 